# 24. Hyperparameter search
Hyperparameters are settings chosen before fitting, unlike coefficients learned from data. Search tools evaluate candidate settings using cross-validation. Grid search enumerates a specified product of choices; randomized search samples a fixed number of candidates. A large search can overfit development evidence, so tuning is not a substitute for a reserved final test.

Put preprocessing inside the searched estimator. Nested parameter names follow step names separated by double underscores. Read mean_test_score, std_test_score, params, and fit times in cv_results_. With refit=True, the selected candidate is fitted again on all supplied development data. Search fits and the final refit are separate costs.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
X, y = load_breast_cancer(return_X_y=True)
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
from sklearn.model_selection import GridSearchCV, StratifiedKFold
cv = StratifiedKFold(3, shuffle=True, random_state=42) # Share reproducible evaluation partitions.
search = GridSearchCV(model, {"logisticregression__C": [0.1, 1, 10]}, cv=cv, scoring="accuracy", refit=True, error_score="raise") # Search the pipeline setting.
search.fit(X_dev, y_dev) # Nine fold fits plus one selected refit.
print("Best settings:", search.best_params_) # Inspect the chosen hyperparameters.
print("Mean CV accuracy:", round(search.best_score_, 3)) # Development selection evidence.
print("Candidates:", len(search.cv_results_["params"])) # Count tried configurations.
assert len(search.cv_results_["params"]) == 3 # Verify the search space.

## Nested cross-validation: evaluate the selection procedure
A search’s best_score_ is the result selected from several noisy candidates. Reporting it as an unbiased post-selection estimate is optimistic. Nested cross-validation puts the search inside an outer evaluation loop: the inner folds choose settings using outer-training data only, and the outer validation fold evaluates that complete choice. This is one valid design; a genuinely untouched final test set can also evaluate a frozen tuned workflow. Neither design fixes leakage or an inappropriate split. Different outer folds may select different settings, so nested CV evaluates the procedure, not one universal hyperparameter value.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_score
X, y = load_wine(return_X_y=True) # Bounded multiclass development data.
inner = StratifiedKFold(3, shuffle=True, random_state=1) # Select settings within each outer-training partition.
outer = StratifiedKFold(3, shuffle=True, random_state=2) # Evaluate the complete selection procedure.
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)) # Keep scaling inside both boundaries.
search = GridSearchCV(pipe, {"logisticregression__C": [0.1, 1.]}, cv=inner, scoring="accuracy", refit=True) # Two candidates, three inner folds.
scores = cross_val_score(search, X, y, cv=outer, scoring="accuracy", error_score="raise") # Each outer validation partition stays out of tuning.
print("Outer accuracies:", scores.round(3)) # Three evaluation results.
print("Mean / spread:", round(scores.mean(), 3), round(scores.std(), 3)) # Descriptive fold summary, not a confidence interval.
print("Fits:", 3 * (2 * 3 + 1)) # Three searches with six candidate fits and one refit each.
assert scores.shape == (3,) # One independent role per outer validation fold.

### Reading the result
There are 21 fits in this bounded design. Fold standard deviation is descriptive because the training sets overlap. After evaluating the process, a final search can fit a deployable model on the permitted development data; the outer scores are not scores of that final fitted instance.

### Transfer exercise
Explain where the scaler, search, and final estimator fit in one outer round. Compare this design with a single reserved final test after tuning.

## Your turn
Count fits for a 20-candidate randomized search with five folds and refit=True. Name one way to reduce computation without using the test set.

Try the task before opening the solution on the course website.